# 🏥 DICOMNET Teachers + MONAI Training
## Lokal C2C + TS Teachers → Colab Eğitimi

Bu notebook ile masaüstünden hazırlanan teacher maskeler ile eğitim:

1. ✅ Drive'dan lokal teacher'ları yükle
2. ✅ Teacher birleştirme ve veri hazırlama
3. ✅ MONAI UNet eğitimi (60 epoch)
4. ✅ Test ve overlay üretimi

---

### 📋 Ön Gereksinimler:
- Masaüstünde `generate_c2c_teachers_local.py` ve `generate_ts_teachers_from_c2c.py` çalıştırılmış
- **C2C_teachers_DICOMNET/**, **TS_teachers_DICOMNET/**, **DICOMNET_nifti/** klasörleri Drive'da
- **Colab GPU runtime** (T4/V100/A100)

### 🚀 Başlamadan Önce:
**Runtime → Change runtime type → GPU** seçin!

---
## 1️⃣ Google Drive Mount + GPU Kontrolü

In [ ]:
# Google Drive mount
from google.colab import drive
import os

drive.mount('/content/drive')

# Çalışma dizini
WORKSPACE = '/content/drive/MyDrive/L3_SO_ANALYSIS'

if os.path.exists(WORKSPACE):
    os.chdir(WORKSPACE)
    print(f"✅ Çalışma dizini: {os.getcwd()}")
else:
    print(f"⚠️  UYARI: {WORKSPACE} bulunamadı!")
    print("Lütfen yukarıdaki WORKSPACE değişkenini düzenleyin.")

In [ ]:
# GPU kontrolü
import torch

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"✅ CUDA Version: {torch.version.cuda}")
else:
    print("❌ GPU bulunamadı! Runtime → Change runtime type → GPU seçin!")

---
## 2️⃣ Paket Kurulumu (5 dakika)

In [ ]:
# Sistem paketleri
!apt-get update -qq
!apt-get install -y -qq libgl1-mesa-glx libglib2.0-0
print("✅ Sistem paketleri kuruldu")

In [ ]:
# PyTorch (CUDA 11.8)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
print("✅ PyTorch kuruldu")

In [ ]:
# Medical imaging ve ML kütüphaneleri
!pip install -q monai[all]>=1.3.0
!pip install -q nibabel pydicom SimpleITK opencv-python-headless
!pip install -q scikit-image scikit-learn matplotlib pandas tqdm
!pip install -q pytorch-lightning
print("✅ MONAI ve medical imaging kütüphaneleri kuruldu")

In [ ]:
# Kurulum doğrulama
import torch
import monai
import nibabel

print("\n📦 Kurulu Paketler:")
print(f"  PyTorch: {torch.__version__}")
print(f"  MONAI: {monai.__version__}")
print(f"  nibabel: {nibabel.__version__}")
print("\n✅ Tüm paketler başarıyla kuruldu!")

---
## 3️⃣ DICOMNET Teacher Yollarını Ayarlama

In [ ]:
# DICOMNET teacher yolları
from pathlib import Path
import json

# Masaüstünden Drive'a yüklenmiş teacher klasörleri
DICOMNET_IMAGES = "/content/drive/MyDrive/DICOMNET_nifti"
C2C_TEACHERS = "/content/drive/MyDrive/C2C_teachers_DICOMNET"
TS_TEACHERS = "/content/drive/MyDrive/TS_teachers_DICOMNET"

# Model ve çıktılar
CHECKPOINT_DIR = "/content/drive/MyDrive/L3_checkpoints"
TEST_OUTPUT_DIR = "/content/drive/MyDrive/L3_test_outputs"
MERGED_DATA_DIR = "/content/drive/MyDrive/MERGED_teachers_DICOMNET"

# Klasörleri oluştur
for folder in [CHECKPOINT_DIR, TEST_OUTPUT_DIR, MERGED_DATA_DIR]:
    os.makedirs(folder, exist_ok=True)

# Veri kontrolü
nifti_images = sorted(Path(DICOMNET_IMAGES).glob("*.nii.gz"))
c2c_manifests = list(Path(C2C_TEACHERS).glob("*/c2c_manifest.json"))
ts_manifests = list(Path(TS_TEACHERS).glob("*/ts_manifest.json"))

print(f"✅ {len(nifti_images)} NIfTI görüntü bulundu")
print(f"✅ {len(c2c_manifests)} C2C teacher vaka bulundu")
print(f"✅ {len(ts_manifests)} TS teacher vaka bulundu")

if len(nifti_images) == 0:
    print(f"\n❌ NIfTI görüntüleri bulunamadı: {DICOMNET_IMAGES}")
    print("⚠️  Lütfen README'deki yükleme adımlarını kontrol edin.")

---
## 4️⃣ Teacher Birleştirme ve Manifest Oluşturma

C2C (VAT/SAT/psoas) + TS (vertebra/muscles/body) → 5 sınıflı mask

In [ ]:
import nibabel as nib
import numpy as np
from tqdm import tqdm

def merge_teachers_for_case(case_name, c2c_dir, ts_dir, image_path, output_dir):
    """
    Tek vaka için C2C + TS teacher'larını birleştir.
    
    Sınıflar:
    0: Background
    1: Vertebra (L3)
    2: Inner abdomen (fascia boundary)
    3: Psoas left
    4: Psoas right
    """
    try:
        # Görüntüyü yükle (referans için)
        img_nib = nib.load(str(image_path))
        img_shape = img_nib.shape
        
        # Boş 5-class mask oluştur
        merged_mask = np.zeros(img_shape, dtype=np.uint8)
        
        # TS vertebra mask (label 1)
        ts_vertebra = Path(ts_dir) / case_name / "vertebrae_L3.nii.gz"
        if ts_vertebra.exists():
            vb_mask = nib.load(str(ts_vertebra)).get_fdata() > 0
            merged_mask[vb_mask] = 1
        
        # C2C psoas masks (label 3, 4)
        c2c_psoas = Path(c2c_dir) / case_name / "psoas.nii.gz"
        if c2c_psoas.exists():
            psoas_data = nib.load(str(c2c_psoas)).get_fdata()
            # Comp2Comp'tan gelen psoas: left=1, right=2 (örnek)
            merged_mask[psoas_data == 1] = 3  # Left
            merged_mask[psoas_data == 2] = 4  # Right
        
        # C2C VAT/SAT → Fascia boundary approximation (label 2)
        c2c_vat = Path(c2c_dir) / case_name / "vat.nii.gz"
        if c2c_vat.exists():
            vat_mask = nib.load(str(c2c_vat)).get_fdata() > 0
            # VAT sınırını fascia boundary olarak kullan (basitleştirilmiş)
            from scipy import ndimage
            vat_dilated = ndimage.binary_dilation(vat_mask, iterations=2)
            fascia_approx = vat_dilated & (~vat_mask)
            merged_mask[fascia_approx] = 2
        
        # Kaydet
        output_path = Path(output_dir) / f"{case_name}_merged.nii.gz"
        merged_nib = nib.Nifti1Image(merged_mask, img_nib.affine)
        nib.save(merged_nib, str(output_path))
        
        return str(output_path)
    
    except Exception as e:
        print(f"⚠️  {case_name} birleştirme hatası: {e}")
        return None

# Tüm vakaları birleştir
merged_manifest = []
for nifti_path in tqdm(nifti_images, desc="Teacher Birleştirme"):
    case_name = nifti_path.stem.replace(".nii", "")
    
    merged_path = merge_teachers_for_case(
        case_name=case_name,
        c2c_dir=C2C_TEACHERS,
        ts_dir=TS_TEACHERS,
        image_path=nifti_path,
        output_dir=MERGED_DATA_DIR
    )
    
    if merged_path:
        merged_manifest.append({
            "image": str(nifti_path),
            "label": merged_path,
            "case_id": case_name
        })

# Manifest kaydet
manifest_path = Path(MERGED_DATA_DIR) / "merged_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(merged_manifest, f, indent=2)

print(f"\n✅ {len(merged_manifest)} vaka birleştirildi")
print(f"✅ Manifest: {manifest_path}")

---
## 5️⃣ Train/Val Split (80/20)

In [ ]:
from sklearn.model_selection import train_test_split

# 80% train, 20% val
train_data, val_data = train_test_split(
    merged_manifest,
    test_size=0.2,
    random_state=42
)

print(f"📊 Train: {len(train_data)} vaka")
print(f"📊 Val: {len(val_data)} vaka")

---
## 6️⃣ MONAI DataLoader + Transforms

In [ ]:
from monai.data import Dataset, DataLoader
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Spacingd, Orientationd,
    ScaleIntensityRanged, RandCropByPosNegLabeld, RandFlipd, RandRotate90d,
    ToTensord, AsDiscreted
)

# Train transforms
train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 3.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=["image"], a_min=-150, a_max=250, b_min=0.0, b_max=1.0, clip=True),
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=(96, 96, 32),
        pos=1,
        neg=1,
        num_samples=2
    ),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandRotate90d(keys=["image", "label"], prob=0.2, max_k=3),
    ToTensord(keys=["image", "label"])
])

# Val transforms (no augmentation)
val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 3.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=["image"], a_min=-150, a_max=250, b_min=0.0, b_max=1.0, clip=True),
    ToTensord(keys=["image", "label"])
])

# DataLoaders
train_ds = Dataset(data=train_data, transform=train_transforms)
val_ds = Dataset(data=val_data, transform=val_transforms)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2)

print("✅ DataLoader hazır")

---
## 7️⃣ MONAI UNet Model

In [ ]:
from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric

# UNet (5 class: background, vertebra, fascia, psoas_L, psoas_R)
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=5,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2
).cuda()

# Loss ve metrik
loss_function = DiceCELoss(to_onehot_y=True, softmax=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
dice_metric = DiceMetric(include_background=False, reduction="mean")

print(f"✅ Model parametreleri: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

---
## 8️⃣ Training Loop (60 Epoch)

⏱️ **Tahmini süre**: ~6-12 saat (GPU'ya göre)

In [ ]:
import time
from monai.inferers import sliding_window_inference

# Training loop
max_epochs = 60
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []

for epoch in range(max_epochs):
    start_time = time.time()
    print(f"\n{'='*50}")
    print(f"Epoch {epoch + 1}/{max_epochs}")
    
    # TRAIN
    model.train()
    epoch_loss = 0
    step = 0
    
    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data["image"].cuda(), batch_data["label"].cuda()
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        if step % 10 == 0:
            print(f"  Train step {step}/{len(train_loader)}, loss: {loss.item():.4f}")
    
    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"  Train loss: {epoch_loss:.4f}")
    
    # VALIDATION
    model.eval()
    with torch.no_grad():
        for val_data in val_loader:
            val_inputs, val_labels = val_data["image"].cuda(), val_data["label"].cuda()
            
            # Sliding window inference (3D)
            val_outputs = sliding_window_inference(
                val_inputs,
                roi_size=(96, 96, 32),
                sw_batch_size=4,
                predictor=model
            )
            
            val_outputs = torch.argmax(val_outputs, dim=1, keepdim=True)
            dice_metric(y_pred=val_outputs, y=val_labels)
    
    metric = dice_metric.aggregate().item()
    dice_metric.reset()
    metric_values.append(metric)
    
    print(f"  Val Dice: {metric:.4f}")
    print(f"  Epoch time: {time.time() - start_time:.1f}s")
    
    # Save best model
    if metric > best_metric:
        best_metric = metric
        best_metric_epoch = epoch + 1
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "best_model.pt"))
        print(f"  ✅ Best model saved! (Dice: {metric:.4f})")
    
    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': epoch_loss,
            'dice': metric
        }, os.path.join(CHECKPOINT_DIR, f"checkpoint_epoch_{epoch+1}.pt"))

print(f"\n{'='*50}")
print(f"✅ Eğitim tamamlandı!")
print(f"En iyi Dice: {best_metric:.4f} (Epoch {best_metric_epoch})")

---
## 9️⃣ Training Metrikleri (Plot)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

# Loss curve
plt.subplot(1, 2, 1)
plt.plot(epoch_loss_values, label="Train Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.legend()
plt.grid(True)

# Dice curve
plt.subplot(1, 2, 2)
plt.plot(metric_values, label="Val Dice")
plt.xlabel("Epoch")
plt.ylabel("Dice Score")
plt.title("Validation Dice")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, "training_curves.png"), dpi=300)
plt.show()

print("✅ Training curves kaydedildi")

---
## 🔟 Test Inference (Örnek Vaka)

En iyi modeli yükleyip bir test vakasında segmentasyon yapıyoruz.

In [ ]:
# Best model yükle
model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, "best_model.pt")))
model.eval()

# Val setinden ilk vakayı test et
test_case = val_data[0]
test_input = val_transforms(test_case)["image"].unsqueeze(0).cuda()

with torch.no_grad():
    test_output = sliding_window_inference(
        test_input,
        roi_size=(96, 96, 32),
        sw_batch_size=4,
        predictor=model
    )
    test_pred = torch.argmax(test_output, dim=1).squeeze().cpu().numpy()

# Overlay oluştur
test_img = nib.load(test_case["image"]).get_fdata()
mid_slice = test_img.shape[2] // 2

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(test_img[:, :, mid_slice], cmap="gray")
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(test_pred[:, :, mid_slice], cmap="jet")
plt.title("Prediction")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(test_img[:, :, mid_slice], cmap="gray")
plt.imshow(test_pred[:, :, mid_slice], cmap="jet", alpha=0.5)
plt.title("Overlay")
plt.axis("off")

plt.tight_layout()
plt.savefig(os.path.join(TEST_OUTPUT_DIR, "test_inference.png"), dpi=300)
plt.show()

print(f"✅ Test inference tamamlandı: {test_case['case_id']}")

---
## ✅ Tamamlandı!

### İndirmeniz Gerekenler:
1. **Best model**: `L3_checkpoints/best_model.pt` → Desktop'a indir
2. **Training curves**: `L3_checkpoints/training_curves.png`
3. **Test overlay**: `L3_test_outputs/test_inference.png`

### Sonraki Adımlar:
- Desktop GUI'de yeni modeli test edin: `python desktop_project/l3_vfa_pma_gui.py --model best_model.pt`
- Radyolog ground truth ile karşılaştırma: `python desktop_project/optimize_params.py --ref ground_truth.csv --model best_model.pt`
- Batch processing: `python desktop_project/batch_process.py --model best_model.pt`